In [ ]:
# ==========================================================
# DRDO Anti-UAV Detection System
# Block 1 : Imports + Paths + GPU Setup
# ==========================================================

import os
import json
import random # randomised operation
import shutil # copy 
from collections import defaultdict # creates default value when new key is accessed

import cv2   #opencv
import numpy as np
import torch 
from tqdm import tqdm

# ==========================================================
# RANDOM SEED
# ==========================================================

SEED = 42

random.seed(SEED)       #python
np.random.seed(SEED)    # numpy
torch.manual_seed(SEED) #pytorch 

# ==========================================================
# DUT DATASET
# ==========================================================

DUT_ROOT = "/kaggle/input/datasets/bninaayoub/dut-anti-uav-tracking-dataset"
#frames of videos
DUT_IMAGE_ROOT = os.path.join(
    DUT_ROOT,
    "Anti-UAV-Tracking-V0",
    "Anti-UAV-Tracking-V0"
)
#anotations
DUT_GT_ROOT = os.path.join(
    DUT_ROOT,
    "Anti-UAV-Tracking-V0GT",
    "Anti-UAV-Tracking-V0GT"
)

# ==========================================================
# COCO DATASET
# ==========================================================

COCO_ROOT = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017"

COCO_IMAGE_ROOT = os.path.join(
    COCO_ROOT,
    "train2017"
)

COCO_JSON = os.path.join(
    COCO_ROOT,
    "annotations",
    "instances_train2017.json"
)

# ==========================================================
# OUTPUT PATHS
# ==========================================================

WORK_DIR = "/kaggle/working"

YOLO_LABELS = os.path.join( # to store labels1.txt , labels2.txt
    WORK_DIR,
    "yolo_labels"
)

FINAL_DATASET = os.path.join( # to save drone folder images labels train val 
    WORK_DIR,
    "drone_dataset"
)

os.makedirs(YOLO_LABELS, exist_ok=True) # continueif  already exists
os.makedirs(FINAL_DATASET, exist_ok=True)

# ==========================================================
# GPU INFORMATION
# ==========================================================

print("="*70)
print("SYSTEM INFORMATION")
print("="*70)

print("PyTorch :", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("="*70)

# ==========================================================
# VERIFY DATASET PATHS
# ==========================================================

assert os.path.exists(DUT_IMAGE_ROOT), "DUT Images not found!"  # if false show the given message
assert os.path.exists(DUT_GT_ROOT), "DUT GT not found!"
assert os.path.exists(COCO_IMAGE_ROOT), "COCO Images not found!"
assert os.path.exists(COCO_JSON), "COCO Annotation not found!"

print("✓ DUT Dataset Loaded")
print("✓ COCO Dataset Loaded")

# ==========================================================
# BASIC STATISTICS
# ==========================================================

dut_videos = sorted(os.listdir(DUT_IMAGE_ROOT)) # sorted video001 video002

total_frames = 0

for v in dut_videos:
    total_frames += len([
        x for x in os.listdir(os.path.join(DUT_IMAGE_ROOT, v)) # inside a video001
        if x.endswith(".jpg")
    ])

print("\nDUT Videos :", len(dut_videos))
print("DUT Frames :", total_frames)

with open(COCO_JSON, "r") as f:
    coco = json.load(f)

print("COCO Train Images :", len(coco["images"]))
print("COCO Categories :", len(coco["categories"]))

print("\nSetup Completed Successfully.")

SYSTEM INFORMATION
PyTorch : 2.10.0+cu128
CUDA Available : True
GPU : Tesla T4
✓ DUT Dataset Loaded
✓ COCO Dataset Loaded

DUT Videos : 20
DUT Frames : 24804
COCO Train Images : 118287
COCO Categories : 80

Setup Completed Successfully.


In [ ]:
# ==========================================================
# Block 2 : Select COCO Negative Images
# ==========================================================

print("="*70)
print("Selecting COCO Negative Images")
print("="*70)

# ---------------------------------------
# COCO Category Mapping
# ---------------------------------------

category_id_to_name = {}
# category_id_to_name =
# {
# 1 : "person",
# 2 : "bicycle",
# 3 : "car",
# 4 : "motorcycle"
# }
for c in coco["categories"]:
    category_id_to_name[c["id"]] = c["name"].lower()

# ---------------------------------------
# Flying classes to REMOVE
# ---------------------------------------

REMOVE_CLASSES = {
    "airplane",
    "bird",
    "kite"
}

print("Excluded Classes :", REMOVE_CLASSES)

# ---------------------------------------
# Image -> Classes Mapping
# ---------------------------------------

# {
#     1 : {"person", "car"},
#     2 : {"dog"},
#     3 : {"bird", "person"},
#     ...
# }

image_to_classes = defaultdict(set)

# coco annotations
# coco["annotations"] =
# [
#     {
#         "image_id":1,
#         "category_id":1
#     },

#     {
#         "image_id":1,
#         "category_id":3
#     },

#     {
#         "image_id":2,
#         "category_id":18
#     }
# ]

for ann in tqdm(coco["annotations"]):

    image_to_classes[ann["image_id"]].add(
        category_id_to_name[ann["category_id"]]
    )

# ---------------------------------------
# Keep only clean negatives
# ---------------------------------------

negative_images = []

# coco_images
# [
#     {
#         "id": 101,
#         "file_name": "000000000101.jpg"
#     },
#     {
#         "id": 102,
#         "file_name": "000000000102.jpg"
#     }
# ]

for img in tqdm(coco["images"]):

    img_id = img["id"]

    classes = image_to_classes.get(img_id, set())

    if len(classes.intersection(REMOVE_CLASSES)) == 0:

        negative_images.append(img)

print("\nTotal COCO Images      :", len(coco["images"]))
print("Negative Candidates    :", len(negative_images))

# ---------------------------------------
# Randomly select negatives
# ---------------------------------------

NUM_NEGATIVE_IMAGES = 4000

random.shuffle(negative_images)

negative_images = negative_images[:NUM_NEGATIVE_IMAGES]

print("Selected Negatives     :", len(negative_images))

# ---------------------------------------
# Preview
# ---------------------------------------

print("\nSample Images")

for img in negative_images[:5]:
    print(img["file_name"])

Selecting COCO Negative Images
Excluded Classes : {'kite', 'airplane', 'bird'}


100%|██████████| 118287/118287 [00:00<00:00, 662607.90it/s]



Total COCO Images      : 118287
Negative Candidates    : 109886
Selected Negatives     : 4000

Sample Images
000000566746.jpg
000000528555.jpg
000000186441.jpg
000000113248.jpg
000000580620.jpg


In [ ]:
# ==========================================================
# Block 3 : Create Final YOLO Dataset
# ==========================================================

from pathlib import Path

FINAL_DATASET = "/kaggle/working/final_yolo_dataset"

if os.path.exists(FINAL_DATASET):        # if already exists delete it completly 
    shutil.rmtree(FINAL_DATASET)

train_img = os.path.join(FINAL_DATASET, "images/train")
val_img   = os.path.join(FINAL_DATASET, "images/val")

train_lbl = os.path.join(FINAL_DATASET, "labels/train")
val_lbl   = os.path.join(FINAL_DATASET, "labels/val")

for p in [train_img, val_img, train_lbl, val_lbl]:
    os.makedirs(p, exist_ok=True)

# --------------------------------------------------
# DUT Split
# --------------------------------------------------

videos = sorted(os.listdir(DUT_IMAGE_ROOT))

TRAIN_VIDEOS = videos[:16]
VAL_VIDEOS   = videos[16:]

print("Train Videos :", TRAIN_VIDEOS)
print("Validation Videos :", VAL_VIDEOS)

dut_train = 0
dut_val = 0

# ----------------------------
# Training Videos
# ----------------------------

for video in tqdm(TRAIN_VIDEOS):

    img_folder = os.path.join(DUT_IMAGE_ROOT, video)
    gt_file = os.path.join(DUT_GT_ROOT, video + "_gt.txt")

    images = sorted([
        x for x in os.listdir(img_folder)
        if x.endswith(".jpg")
    ])

    with open(gt_file) as f:
        anns = f.readlines()

    for img_name, ann in zip(images, anns):

        img_path = os.path.join(img_folder, img_name)

        img = cv2.imread(img_path)
        H, W = img.shape[:2]

        x, y, w, h = map(float, ann.split())

        xc = (x + w/2) / W
        yc = (y + h/2) / H
        ww = w / W
        hh = h / H

        shutil.copy(
            img_path,
            os.path.join(train_img, f"{video}_{img_name}")
        )

        label_name = f"{video}_{Path(img_name).stem}.txt"

        with open(os.path.join(train_lbl, label_name), "w") as out:
            out.write(f"0 {xc:.6f} {yc:.6f} {ww:.6f} {hh:.6f}")

        dut_train += 1

# ----------------------------
# Validation Videos
# ----------------------------

for video in tqdm(VAL_VIDEOS):

    img_folder = os.path.join(DUT_IMAGE_ROOT, video)         # image
    gt_file = os.path.join(DUT_GT_ROOT, video + "_gt.txt")   # labels

    images = sorted([                                        
        x for x in os.listdir(img_folder)
        if x.endswith(".jpg")
    ])

    with open(gt_file) as f:
        anns = f.readlines()
    #         anns =
    # [
    # "120 80 40 30\n",
    # "130 82 41 31\n",
    # ...
    # ]

    for img_name, ann in zip(images, anns):

        img_path = os.path.join(img_folder, img_name)

        img = cv2.imread(img_path)   #Reads the image into memory.
        H, W = img.shape[:2]

        x, y, w, h = map(float, ann.split()) 
      
        xc = (x + w/2) / W
        yc = (y + h/2) / H
        ww = w / W
        hh = h / H

        shutil.copy(
            img_path,
            os.path.join(val_img, f"{video}_{img_name}") # copy to validation folder 
        )

        label_name = f"{video}_{Path(img_name).stem}.txt" # creatng new file to save normalised labels

        with open(os.path.join(val_lbl, label_name), "w") as out:
            #class_id x_center y_center width height
            out.write(f"0 {xc:.6f} {yc:.6f} {ww:.6f} {hh:.6f}")  # 0 for class drone is the only video

        dut_val += 1

# --------------------------------------------------
# COCO Negatives
# --------------------------------------------------

split = int(0.8 * len(negative_images))

train_neg = negative_images[:split]
val_neg = negative_images[split:]

for img in tqdm(train_neg):

    src = os.path.join(COCO_IMAGE_ROOT, img["file_name"])

    dst = os.path.join(train_img, img["file_name"])

    shutil.copy(src, dst)

    open(            # an empty file created for negative data 
        os.path.join(
            train_lbl,
            Path(img["file_name"]).stem + ".txt"
        ),
        "w"
    ).close()

for img in tqdm(val_neg):

    src = os.path.join(COCO_IMAGE_ROOT, img["file_name"])

    dst = os.path.join(val_img, img["file_name"])

    shutil.copy(src, dst)

    open(
        os.path.join(
            val_lbl,
            Path(img["file_name"]).stem + ".txt"
        ),
        "w"
    ).close()

# --------------------------------------------------
# data.yaml
# --------------------------------------------------
# for info of yolo
yaml_text = f"""
path: {FINAL_DATASET}

train: images/train
val: images/val

names:
  0: drone
"""
#writing to yaml 
with open(os.path.join(FINAL_DATASET, "data.yaml"), "w") as f:
    f.write(yaml_text)

print("\n==============================")
print("Dataset Ready")
print("==============================")
print("DUT Train :", dut_train)
print("DUT Val   :", dut_val)
print("COCO Train:", len(train_neg))
print("COCO Val  :", len(val_neg))
print("Total Train :", dut_train + len(train_neg))
print("Total Val   :", dut_val + len(val_neg))

Train Videos : ['video01', 'video02', 'video03', 'video04', 'video05', 'video06', 'video07', 'video08', 'video09', 'video10', 'video11', 'video12', 'video13', 'video14', 'video15', 'video16']
Validation Videos : ['video17', 'video18', 'video19', 'video20']


100%|██████████| 800/800 [00:04<00:00, 162.42it/s]


Dataset Ready
DUT Train : 19769
DUT Val   : 5035
COCO Train: 3200
COCO Val  : 800
Total Train : 22969
Total Val   : 5835


In [ ]:
# ==========================================================
# Block 4 : Production YOLO11 Training
# ==========================================================

!pip -q install -U ultralytics

import ultralytics          # contains implementation of yolov11
from ultralytics import YOLO
import torch

print("="*70)
print("Environment")
print("="*70)

print("Ultralytics :", ultralytics.__version__)
print("PyTorch     :", torch.__version__)
print("CUDA        :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("="*70)

# --------------------------------------------------
# Load pretrained YOLO11 Nano
# --------------------------------------------------

model = YOLO("yolo11n.pt")

# --------------------------------------------------
# Train
# --------------------------------------------------

results = model.train(

    # Dataset
    data="/kaggle/working/final_yolo_dataset/data.yaml",

    # Training
    epochs=50,
    patience=10,

    imgsz=640,
    batch=16,

    device=0,
    workers=2,   # img load

    pretrained=True,

    # Optimizer
    optimizer="AdamW",

    lr0=0.001,
    lrf=0.01,

    momentum=0.937,
    weight_decay=0.0005, #penalty to reduce overfitting

    warmup_epochs=3,

    # Augmentation
    hsv_h=0.015,  # Hue (color)
    hsv_s=0.70,   #Saturation(color-intensity)
    hsv_v=0.40,   #Value(brightness)

    translate=0.10, # drone left right 
    scale=0.50,    # zoom in zoom out  

    fliplr=0.5,    # direction of motion flipped horizontally
    flipud=0.0,

    mosaic=1.0,   # 4 different image in 1 small obj detection
    mixup=0.10,   # 2 image blend together 

    degrees=5,    # rotates by 5 deg

    # Validation
    val=True,

    plots=True,

    # Saving
    save=True,
    save_period=5,

    project="/kaggle/working",
    name="YOLO11_DRDO",

    exist_ok=True,

    # Performance
    amp=True,  #(Automatic Mixed Precision) traim
    cache=False,

    # Reproducibility
    seed=42,
    deterministic=True,

    verbose=True             # logs
)

print("\nTraining Finished Successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.3 MB/s eta 0:00:0000:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Environment
Ultralytics : 8.4.92
PyTorch     : 2.10.0+cu128
CUDA        : True
GPU : Tesla T4
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/final_yolo_dataset/dat